# 07 — Baseline Dispatch Visualisation

**Purpose:** Two-panel figure for the baseline E4ST scenario:

1. **Dispatch stack** — stacked-area chart of annual generation (GWh) by fuel
   type across BA zones sorted by marginal LMP (cheapest → dearest, left → right).
   This is the closest analogue to a dispatch-curve for a single-period zonal model.
   Fuel colours are the same scheme used in `01_eia_pull.ipynb`.

2. **Renewable-penetration heatmap** — rows = 67 BA zones sorted by renewable
   share, columns = fuel types, cells = fraction of each BA's annual generation
   from that fuel.  Reveals which BAs are most renewable-heavy and what mix
   drives that.  Note: E4ST results are annual-average aggregates — no
   sub-annual temporal dimension is available in the output files.

**Inputs:**
- `data/processed/e4st_results/baseline/dispatch.parquet`
- `data/processed/e4st_results/baseline/lmp.parquet`

**Outputs:**
- `data/processed/figures/dispatch_baseline.png` (300 DPI)

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker

FIGURES_DIR = PROJECT_ROOT / 'data' / 'processed' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Fuel colour scheme — identical to 01_eia_pull FUEL_COLORS, extended ───────
# Keys match `genfuel` column values in dispatch.parquet
FUEL_COLORS = {
    'coal':       '#2d2d2d',   # black  (01: 'coal')
    'ng':         '#4169e1',   # blue   (01: 'natural gas')
    'nuclear':    '#d62728',   # red    (01: 'nuclear')
    'wind':       '#2ca02c',   # green  (01: 'wind')
    'solar':      '#ff7f0e',   # orange (01: 'solar')
    'hydro':      '#17becf',   # cyan   (01: 'hydro')
    'biomass':    '#8c564b',   # brown
    'geothermal': '#e377c2',   # pink
    'oil':        '#9467bd',   # purple
    'storage':    '#bcbd22',   # yellow-green
    'other':      '#7f7f7f',   # gray
}

# Stacking order: fossils first (bottom), then clean (top)
FUEL_ORDER = [
    'coal', 'oil', 'ng', 'other', 'storage',
    'biomass', 'geothermal', 'hydro', 'nuclear', 'solar', 'wind'
]

# Renewable/clean set for penetration calculation
CLEAN_FUELS = {'wind', 'solar', 'hydro', 'nuclear', 'geothermal', 'biomass'}

In [ ]:
# ── Load baseline dispatch and LMP ────────────────────────────────────────────
BASE_DIR = PROJECT_ROOT / 'data' / 'processed' / 'e4st_results' / 'baseline'

dispatch = pd.read_parquet(BASE_DIR / 'dispatch.parquet')
lmp      = pd.read_parquet(BASE_DIR / 'lmp.parquet')

# Convert MWh → GWh for panel 1
dispatch['dispatch_gwh'] = dispatch['dispatch_mwh'] / 1e6

fuels_present = [f for f in FUEL_ORDER if f in dispatch['genfuel'].unique()]
print('Fuel types present:', fuels_present)
print('BAs:', dispatch['ba'].nunique())

In [ ]:
# ── Prepare panel-1 data: BA-level dispatch pivot ─────────────────────────────
# Pivot: rows=BA, columns=fuel, values=GWh; fill missing fuel-BA combos with 0
ba_gen = (
    dispatch
    .groupby(['ba', 'genfuel'])['dispatch_gwh']
    .sum()
    .unstack(fill_value=0.0)
    .reindex(columns=fuels_present, fill_value=0.0)
)

# Sort BAs by their zone's mean LMP (ascending = cheapest-dispatch first)
lmp_sorted = lmp.set_index('ba')['lmp_mwh'].sort_values()
ba_gen = ba_gen.reindex(lmp_sorted.index).dropna(how='all')

print('BA dispatch pivot shape:', ba_gen.shape)
ba_gen.head(3)

In [ ]:
# ── Prepare panel-2 data: fuel-share heatmap ──────────────────────────────────
# Normalise each BA row to 100 % to get generation share
ba_share = ba_gen.div(ba_gen.sum(axis=1).replace(0, np.nan), axis=0) * 100

# Sort by descending renewable share (most renewable BAs at top)
ba_share['_ren'] = ba_share[
    [f for f in CLEAN_FUELS if f in ba_share.columns]
].sum(axis=1)
ba_share = ba_share.sort_values('_ren', ascending=False).drop(columns='_ren')

# Column order for heatmap: renewables on left, fossils on right
heatmap_col_order = [
    'wind', 'solar', 'hydro', 'nuclear', 'geothermal', 'biomass',
    'storage', 'other', 'ng', 'oil', 'coal'
]
heatmap_cols = [f for f in heatmap_col_order if f in ba_share.columns]
ba_share = ba_share[heatmap_cols]

print('Heatmap shape:', ba_share.shape)

In [ ]:
# ── Figure: two-panel layout ───────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 12), constrained_layout=False)
fig.subplots_adjust(left=0.07, right=0.97, top=0.93, bottom=0.07, hspace=0.35)

ax1 = fig.add_subplot(2, 1, 1)
ax2 = fig.add_subplot(2, 1, 2)

# ── Panel 1: Stacked area / dispatch stack ─────────────────────────────────────
x = np.arange(len(ba_gen))
ys = [ba_gen[f].values for f in fuels_present]
colors = [FUEL_COLORS[f] for f in fuels_present]

ax1.stackplot(x, ys, labels=fuels_present, colors=colors, alpha=0.88)

# x-tick labels: BA codes every 5 zones to avoid crowding
tick_step = max(1, len(ba_gen) // 20)
ax1.set_xticks(x[::tick_step])
ax1.set_xticklabels(ba_gen.index[::tick_step], rotation=55, ha='right', fontsize=7.5)
ax1.set_xlim(0, len(ba_gen) - 1)
ax1.set_ylabel('Annual Generation (GWh)', fontsize=10)
ax1.set_xlabel('Balancing Authority (sorted by increasing LMP →)', fontsize=9)
ax1.set_title('(a)  Baseline Dispatch Stack by Fuel — Sorted by Zonal LMP', fontsize=11, loc='left')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax1.spines[['top', 'right']].set_visible(False)

# Legend: reverse so top-of-stack fuel is first
handles, labels = ax1.get_legend_handles_labels()
ax1.legend(
    handles[::-1], labels[::-1],
    ncol=2, loc='upper left', frameon=False, fontsize=8.5,
    borderaxespad=0.4,
)

# ── Panel 2: Renewable-penetration heatmap ─────────────────────────────────────
im = ax2.imshow(
    ba_share.values,
    aspect='auto',
    cmap='YlGn',
    vmin=0,
    vmax=100,
    interpolation='nearest',
)

ax2.set_xticks(range(len(heatmap_cols)))
ax2.set_xticklabels(heatmap_cols, rotation=35, ha='right', fontsize=9)
ax2.set_yticks(range(len(ba_share)))
ax2.set_yticklabels(ba_share.index, fontsize=6)
ax2.set_title(
    '(b)  Generation Mix by BA — Annual Average Share (%)\n'
    '     Rows sorted by descending clean-energy fraction; '
    'columns: renewables left, fossils right',
    fontsize=11, loc='left'
)

# Colorbar
cbar = fig.colorbar(im, ax=ax2, fraction=0.015, pad=0.01)
cbar.set_label('Share of BA generation (%)', fontsize=9)
cbar.ax.tick_params(labelsize=8)

# Add vertical divider between clean and fossil columns
n_clean = sum(1 for f in heatmap_cols if f in CLEAN_FUELS)
ax2.axvline(n_clean - 0.5, color='white', linewidth=2, linestyle='--', alpha=0.7)
ax2.text(n_clean - 0.5, -1.8, '← clean  |  fossil →',
         ha='center', va='top', fontsize=7.5, color='#333333')

out = FIGURES_DIR / 'dispatch_baseline.png'
fig.savefig(out, dpi=300, bbox_inches='tight')
print(f'Saved \u2192 {out}')
plt.show()